# 4. Agentic RAG (Advanced) — 제품 탑재형 AI: 매뉴얼/트러블슈팅 RAG
**Dynamic Tool Generation + Hybrid Retrieval(Vector + BM25) + 출처 강제 + Self-verification**

---

## 🧩 시나리오(가상)
스마트 생활가전/로보틱스 제품에 탑재된 AI는 사용자의 질문에 답할 때,
반드시 **제품 매뉴얼/트러블슈팅 가이드** 근거를 기반으로 답해야 합니다.

- “세탁기 5C 에러는 뭐예요?”
- “에어컨 필터 경고 뜨면 어떻게 해요?”
- “로봇청소기 도킹이 안 돼요”

이 노트북은 더미 매뉴얼 문서를 생성하고, **RAG + 에이전트**로 근거 기반 답변을 만드는 실습입니다.


In [1]:
!pip -q install -U langchain langchain-community langchain-openai langchain-text-splitters chromadb rank-bm25 pydantic

ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/conda/lib/python3.10/site-packages/typing_extensions-4.15.0.dist-info/licenses/LICENSE'
Consider using the `--user` option or check the permissions.



In [2]:
import os, getpass, glob, textwrap
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain.tools import tool
from langchain.agents import create_agent


PROXY_URL = "http://10.0.4.135:8000/v1" 
PROXY_TOKEN = "1b2d4df88f5d71a9c55617796faf99f1" 



/opt/conda/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) 더미 매뉴얼 문서 생성

기본 실습보다 문서를 2개 더 추가합니다(보안/보상).  


In [3]:
def create_manual_files():
    manuals = {
        "manual_robot_vacuum.txt": """[로봇청소기 사용자 가이드(더미)]
- 도킹이 안 될 때:
  1) 도킹 스테이션 앞 1m 이내 장애물을 제거하세요.
  2) 충전 단자에 먼지가 있으면 마른 천으로 닦으세요.
  3) 지도/맵이 손상되었을 수 있으니 '맵 재설정'을 고려하세요(초기화 아님).
- 충돌(bump)이 많을 때:
  - 조명/거울/유광 바닥에서 센서 오인식이 발생할 수 있습니다.
""",
        "manual_washing_machine.txt": """[세탁기 트러블슈팅 가이드(더미)]
- 에러코드 5C(배수):
  1) 전원 OFF 후 배수 필터를 열고 이물질을 제거하세요.
  2) 배수 호스가 꺾이거나 막히지 않았는지 확인하세요.
  3) 물이 남아 있으면 응급 배수 호스를 사용하세요.
  - 위 조치 후에도 반복되면 서비스 점검이 필요할 수 있습니다.
""",
        "manual_air_conditioner.txt": """[에어컨 가이드(더미)]
- 필터 경고:
  - 필터를 분리해 미지근한 물로 세척 후 완전히 건조하세요.
  - 필터 장착 후 '필터 리셋'을 실행하세요(모델에 따라 메뉴 경로 상이).
- 권장 온도:
  - 냉방 24~26도 권장(환경에 따라 다름).
""",
        "manual_refrigerator.txt": """[냉장고 가이드(더미)]
- 문 열림 경고:
  - 문이 완전히 닫혔는지 확인하고, 도어 패킹(고무) 이물질을 제거하세요.
- 이상 소음:
  - 수평이 맞지 않으면 진동음이 증가할 수 있습니다. 수평 조절 다리를 확인하세요.
""",
    }

    for name, content in manuals.items():
        with open(name, "w", encoding="utf-8") as f:
            f.write(content)

    print("✅ 생성 완료:", ", ".join(manuals.keys()))

create_manual_files()





✅ 생성 완료: manual_robot_vacuum.txt, manual_washing_machine.txt, manual_air_conditioner.txt, manual_refrigerator.txt


## 2) Dynamic Tool Generation + Hybrid Retriever

- 각 txt 파일을 읽고
- 문서명을 기반으로 collection/tool을 자동 생성합니다.

Hybrid Retrieval:
- Vector Retriever (Chroma)
- BM25 Retriever
- EnsembleRetriever로 결합 (가중치는 실습에서 조정 가능)


In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=60)
emb = OpenAIEmbeddings(base_url=PROXY_URL, api_key=PROXY_TOKEN)

def build_retrievers_for_file(path: str):
    docs = TextLoader(path, encoding="utf-8").load()
    chunks = splitter.split_documents(docs)

    # Vector store
    collection = os.path.splitext(os.path.basename(path))[0]
    vectordb = Chroma.from_documents(chunks, embedding=emb, collection_name=f"col_{collection}")
    vector_retriever = vectordb.as_retriever(search_kwargs={"k": 3})

    # BM25
    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = 3

    # Ensemble
    ensemble = EnsembleRetriever(retrievers=[vector_retriever, bm25], weights=[0.6, 0.4])
    return ensemble

manual_files = sorted(glob.glob("manual_*.txt"))
retrievers: Dict[str, Any] = {}
for f in manual_files:
    key = os.path.splitext(os.path.basename(f))[0].replace("manual_", "")
    retrievers[key] = build_retrievers_for_file(f)

print("✅ 카테고리:", retrievers.keys())


✅ 카테고리: dict_keys(['air_conditioner', 'air_purifier', 'refrigerator', 'robot_vacuum', 'washing_machine'])


## 3) Retriever를 Tool로 자동 래핑

- tool 이름: `search_<category>`
- 반환: 검색된 chunk를 **출처 태그**와 함께 반환하여 citation에 활용합니다.


In [5]:
tools = []


def make_search_tool(category: str):
    def _search(query: str) -> str:
        docs = retrievers[category].invoke(query)
        if not docs:
            return "검색 결과가 없습니다."
        return "\n\n".join(d.page_content for d in docs[:4])

    _search.__name__ = f"search_{category}"
    _search.__doc__ = (
        f"제품 매뉴얼/가이드 중 '{category}' 카테고리를 검색합니다. "
        f"{category} 관련 질문일 때 사용하고, 반환된 텍스트를 근거로 답변합니다."
    )
    return tool(_search)



for cat in retrievers.keys():
    tools.append(make_search_tool(cat))

[t.name for t in tools]


['search_air_conditioner',
 'search_air_purifier',
 'search_refrigerator',
 'search_robot_vacuum',
 'search_washing_machine']

## 4) Agent 구성: 출처 기반 답변 + Self-verification

규칙:
1) 질문에 필요한 카테고리 도구를 **1개 이상** 호출한다.
2) 최종 답변에는 항목마다 **출처 태그**를 포함한다.
3) 출처 없이 단정할 수 없으면 "매뉴얼집에 없음"이라고 말한다.
4) 마지막에 self-check: 답변에 출처가 누락되었으면 스스로 수정한다.


In [6]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, base_url=PROXY_URL, api_key=PROXY_TOKEN)

SYSTEM = """당신은 '제품 매뉴얼/트러블슈팅' 가이드 AI입니다.
규칙:
- 반드시 tools로 검색한 내용만 근거로 답변하세요.
- 답변에는 근거 출처 태그([category:chunkN])를 항목마다 포함하세요.
- 매뉴얼에 없는 내용은 '매뉴얼/가이드에 없음'이라고 답하세요.
- 사용자가 여러 기기/카테고리를 묻으면 필요한 도구를 모두 호출하세요.
"""

agent = create_agent(llm, tools, system_prompt=SYSTEM)

def answer_with_self_check(question: str) -> str:
    res = agent.invoke({"messages": [("user", question)]})
    draft = res["messages"][-1].content

    # Self-check: 출처 태그가 없으면 한 번 더 수정 요청
    if "[" not in draft or ":chunk" not in draft:
        fix_prompt = f"""다음 답변은 출처 태그가 부족합니다. 각 주장/항목마다 반드시 [category:chunkN] 출처를 포함하도록 수정하세요.

[질문]
{question}

[초안]
{draft}
"""
        res2 = agent.invoke({"messages": [("user", fix_prompt)]})
        return res2["messages"][-1].content
    return draft

# 예시 실행(비용 발생)
print(answer_with_self_check("세탁기 5C 에러가 뜨는데 어떻게 해야 해? 근거 포함해서 알려줘.")) 


세탁기에서 5C 에러가 발생했을 경우, 다음과 같은 조치를 취해보세요:

1. 전원을 OFF 한 후 배수 필터를 열고 이물질을 제거하세요.
2. 배수 호스가 꺾이거나 막히지 않았는지 확인하세요.
3. 물이 남아 있다면 응급 배수 호스를 사용하세요.

위의 조치를 취한 후에도 문제가 지속된다면 서비스 점검이 필요할 수 있습니다. [category:chunk1]


## 🧪 간단 점검: Retriever 단독 테스트

LLM을 호출하지 않고도, 각 retriever가 적절한 chunk를 찾는지 먼저 확인할 수 있습니다.


In [7]:
# LLM 없이 retriever만 테스트
print(retrievers["washing_machine"].invoke("5C 배수")[0].page_content)
print(retrievers["robot_vacuum"].invoke("도킹이 안 될 때")[0].page_content)


[세탁기 트러블슈팅 가이드(더미)]
- 에러코드 5C(배수):
  1) 전원 OFF 후 배수 필터를 열고 이물질을 제거하세요.
  2) 배수 호스가 꺾이거나 막히지 않았는지 확인하세요.
  3) 물이 남아 있으면 응급 배수 호스를 사용하세요.
  - 위 조치 후에도 반복되면 서비스 점검이 필요할 수 있습니다.
[로봇청소기 사용자 가이드(더미)]
- 도킹이 안 될 때:
  1) 도킹 스테이션 앞 1m 이내 장애물을 제거하세요.
  2) 충전 단자에 먼지가 있으면 마른 천으로 닦으세요.
  3) 지도/맵이 손상되었을 수 있으니 '맵 재설정'을 고려하세요(초기화 아님).
- 충돌(bump)이 많을 때:
  - 조명/거울/유광 바닥에서 센서 오인식이 발생할 수 있습니다.


## 📝 실습 과제 

### 문제 1) 새 문서 추가 자동 반영 (중)
- `manual_remote.txt`(재택근무 매뉴얼)를 새로 만들고,
- 위 Dynamic Tool Generation 코드 수정 없이(또는 최소 수정으로) 자동으로 tool이 생성되게 하세요.

### 문제 2) Citation 강제 포맷 강화 (상)
- 답변을 아래 포맷으로 강제해보세요.
  - 항목별로: `- 결론 ... (근거: [category:chunkN])`
- Self-check에서 포맷이 깨지면 재작성하도록 강화하세요.

### 문제 3) 간단 평가셋 만들기 (상)
- 질문 8개 정도를 만들고,
- 각 질문에 대해 "필수로 호출돼야 하는 카테고리"를 적어둔 뒤,
- agent 실행 로그(messages)를 분석해 tool 호출이 맞는지 점검하는 함수를 작성해보세요.

> 아래 셀은 학생용 TODO 입니다.


In [8]:
# ✅ TODO 셀 (학생용)
# 1) 새로운 매뉴얼 파일을 추가해보세요. (예: manual_air_purifier.txt)
# 2) 파일을 추가한 뒤, manual_files/retrievers/tools를 재생성하여 자동 반영되는지 확인하세요.
# 3) Self-check를 강화해보세요:
#    - 답변에 출처 태그가 '2개 이상' 없으면 수정 요청
#    - "매뉴얼에 없음"이라고 말할 때도, 검색 결과가 비어있었음을 근거로 남기기

# raise NotImplementedError("학생용 TODO 셀입니다. 아래 '참고 답안'을 보기 전까지 구현해보세요.")


In [9]:
# ✅ 참고 답안

ap_doc = """[공기청정기 가이드(더미)]
- 필터 교체:
  - 필터 교체 주기는 사용 환경에 따라 다릅니다. 필터 경고가 뜨면 교체를 권장합니다.
- 냄새가 날 때:
  - 필터 오염/습기 원인이 있을 수 있습니다. 환기 후 필터 상태를 확인하세요.
"""
with open("manual_air_purifier.txt", "w", encoding="utf-8") as f:
    f.write(ap_doc)

# 자동 반영을 위해 manual_files를 다시 스캔하고 retrievers/tools를 재생성
manual_files = sorted(glob.glob("manual_*.txt"))
retrievers = {}
for f in manual_files:
    key = os.path.splitext(os.path.basename(f))[0].replace("manual_", "")
    retrievers[key] = build_retrievers_for_file(f)

tools = [make_search_tool(cat) for cat in retrievers.keys()]
print("✅ 새 카테고리:", retrievers.keys())
print("✅ tools:", [t.name for t in tools])


✅ 새 카테고리: dict_keys(['air_conditioner', 'air_purifier', 'refrigerator', 'robot_vacuum', 'washing_machine'])
✅ tools: ['search_air_conditioner', 'search_air_purifier', 'search_refrigerator', 'search_robot_vacuum', 'search_washing_machine']


In [10]:
# agent도 새 tools로 다시 생성
SYSTEM_V2 = """당신은 '제품 매뉴얼/트러블슈팅' 가이드 AI입니다.
규칙:
- 반드시 tools로 검색한 내용만 근거로 답변하세요.
- 각 항목은 반드시 다음 형식을 따르세요:
  - 결론 ... (근거: [category:chunkN])
- 답변 전체에 출처 태그가 최소 2개 이상 있어야 합니다.
- 매뉴얼에 없는 내용은 '매뉴얼/가이드에 없음'이라고 답하되,
  검색 결과가 비어있었음을 근거와 함께 설명하세요.
- 사용자가 여러 기기/카테고리를 묻으면 필요한 도구를 모두 호출하세요.
"""

agent = create_agent(llm, tools, system_prompt=SYSTEM_V2)


In [11]:
def answer_with_self_check_v2(question: str) -> str:
    res = agent.invoke({"messages": [("user", question)]})
    draft = res["messages"][-1].content

    citation_count = draft.count(":chunk")

    bad_format = "- " not in draft or "(근거: [" not in draft
    not_enough_citations = citation_count < 2
    missing_empty_evidence = ("매뉴얼/가이드에 없음" in draft) and ("검색 결과" not in draft)

    if bad_format or not_enough_citations or missing_empty_evidence:
        fix_prompt = f"""다음 답변은 규칙을 만족하지 못했습니다. 아래 조건을 모두 만족하도록 다시 작성하세요.

조건:
- 각 항목은 '- 결론 ... (근거: [category:chunkN])' 형식
- 출처 태그는 최소 2개 이상
- '매뉴얼/가이드에 없음'이라고 답할 때는 검색 결과가 비어있었음을 함께 근거로 설명

[질문]
{question}

[초안]
{draft}
"""
        res2 = agent.invoke({"messages": [("user", fix_prompt)]})
        return res2["messages"][-1].content

    return draft



In [12]:
print("새 카테고리:", list(retrievers.keys()))
print("tools:", [t.name for t in tools])

새 카테고리: ['air_conditioner', 'air_purifier', 'refrigerator', 'robot_vacuum', 'washing_machine']
tools: ['search_air_conditioner', 'search_air_purifier', 'search_refrigerator', 'search_robot_vacuum', 'search_washing_machine']


In [13]:
print(answer_with_self_check_v2("공기청정기에서 냄새가 나요. 어떻게 해야 하나요?"))
print(answer_with_self_check_v2("공기청정기 필터는 언제 교체해야 하나요?"))
print(answer_with_self_check_v2("재택근무 장비 VPN 설정 방법 알려줘"))


- 결론 공기청정기에서 냄새가 나는 경우, 필터 오염이나 습기가 원인일 수 있습니다. 따라서, 먼저 환기를 한 후 필터 상태를 확인하는 것이 좋습니다. 필터 경고가 뜨면 교체를 권장합니다. (근거: [air_purifier:chunk1]) 
- 결론 필터를 정기적으로 점검하고 교체하는 것이 중요합니다. 사용 환경에 따라 필터 교체 주기가 달라질 수 있으므로 주의가 필요합니다. (근거: [air_purifier:chunk1]) 

출처: 공기청정기 가이드
- 결론 공기청정기 필터는 사용 환경에 따라 교체 주기가 다르며, 필터 경고가 뜨면 교체를 권장합니다. (근거: [air_purifier:chunk1])
- 결론 필터에서 냄새가 날 경우, 필터 오염이나 습기가 원인일 수 있으므로 환기 후 필터 상태를 확인해야 합니다. (근거: [air_purifier:chunk1])

출처: [air_purifier:chunk1]
- 결론: 재택근무 장비의 VPN 설정 방법에 대한 정보는 매뉴얼/가이드에 없습니다. 여러 카테고리에서 VPN 설정에 대한 내용을 검색했으나, 관련된 정보가 발견되지 않았습니다. (근거: [category:air_conditioner], [category:air_purifier], [category:refrigerator], [category:robot_vacuum], [category:washing_machine])
